In [269]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import make_scorer, mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
import statsmodels.api as sm
import math as m
import cvxpy as cp

In [236]:
# load data
df_att = pd.read_csv('attendance data with features.csv')

In [262]:
# preliminary feature selection
# remove features that have obvious (linear) dependencies
# remove features that have no relevance to attendance

df_att_feat = df_att.drop(columns = ['HGP', 'HAvAge', 'VGP', 'VAvAge'])
df_att_feat = df_att_feat.drop(columns = ['HRk', 'VRk'])
df_att_feat = df_att_feat.drop(columns = ['HOL', 'VOL'])
df_att_feat = df_att_feat.drop(columns = ['HGF', 'VGF', 'HGA', 'VGA'])
df_att_feat = df_att_feat.drop(columns = ['HSRS', 'VSRS', 'HPTS%', 'VPTS%', 
                                    'HPTS', 'VPTS', 'HGF/G', 'VGF/G', 
                                    'HGA/G', 'VGA/G', 'HPP', 'VPP',
                                    'HPPO', 'VPPO', 'HPPA', 'VPPA',
                                    'HPPOA', 'VPPOA', 'HS', 'VS', 
                                    'HSA', 'VSA'])

In [263]:
# to my mind
# using PA, LPA, or LA/LC will be most useful
# will run regressions on all, use one with least test error

In [264]:
### FEATURE SELECTION
### VIF
### lovely, basically all features are agressively colinear

# Assuming df is your DataFrame with features X (excluding the target column y)
X = df_att_feat.drop(columns = ['A', 'LA', 'PA', 'LPA', 'LA/LC', 'H', 'V'])  # Replace 'A' with your target column name

# calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data['feature'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X, i) for i in range(X.shape[1])]

vif_data[vif_data['VIF'] <= 10]

,feature,VIF
2,HSOW,6.311005
3,HSOL,7.360741
4,HSOS,1.837430
7,HSH,6.767979
8,HSHA,8.626076
13,HSO,8.980652
14,HCAN,2.101442
17,VSOW,5.615656
18,VSOL,6.603865
19,VSOS,1.726599


In [265]:
# Assuming df is your DataFrame with features X (excluding the target column y)
X = df_att.drop(columns = ['A', 'LA', 'PA', 'LPA', 'LA/LC', 'H', 'V'])  # Replace 'A' with your target column name

# Initialize PCA, you can specify the number of components
pca = PCA(n_components = 5)  # Keep the first 2 principal components

# Fit PCA on the standardized data
X_pca = pca.fit_transform(X)

# Create a DataFrame with the new PCA components
pca_df = pd.DataFrame(X_pca)

print(pca_df.head())

# To check how much variance each component explains:
print(f"Explained variance by each component: {pca.explained_variance_ratio_}")

             0           1           2           3           4
0  -688.380428 -328.808521  395.916954    8.680536  272.710067
1   -20.900219 -568.198117  219.007031    8.990798   30.249943
2 -1478.915572  388.450647 -207.604057   25.745104  201.760468
3  -388.235939 -376.483169  -56.655313 -498.146677 -158.164137
4  -785.380919 -144.255706 -108.887266 -231.712387 -151.823733
Explained variance by each component: [0.49036696 0.45037707 0.02479351 0.02035645 0.01064479]


In [266]:
# # calculate normalized attendance MSE --> so convert everything back to raw attendance numbers
def calc_mse_att(dep, cap_train, cap_test, y_train, y_test, y_train_pred, y_test_pred):
    if dep == 'A':
        y_train_pred_att = y_train_pred
        y_test_pred_att = y_test_pred

        y_train_att = y_train
        y_test_att = y_test
    
    elif dep == 'LA':
        y_train_pred_att = np.exp(y_train_pred)
        y_test_pred_att = np.exp(y_test_pred)

        y_train_att = np.exp(y_train)
        y_test_att = np.exp(y_test)

    elif dep == 'PA':
        y_train_pred_att = y_train_pred * cap_train
        y_test_pred_att = y_test_pred * cap_test

        y_train_att = y_train * cap_train
        y_test_att = y_test * cap_test

    elif dep == 'LPA':
        y_train_pred_att = np.exp(y_train_pred) * cap_train
        y_test_pred_att = np.exp(y_test_pred) * cap_test

        y_train_att = np.exp(y_train) * cap_train
        y_test_att = np.exp(y_test) * cap_test

    elif dep == 'LA/LC':
        lcap_train = m.log(cap_train)
        lcap_test = m.log(cap_test)
        
        y_train_pred_att = np.exp(y_train_pred * lcap_train)
        y_test_pred_att = np.exp(y_test_pred * lcap_test)

        y_train_att = np.exp(y_train * lcap_train)
        y_test_att = np.exp(y_test * lcap_test)
        
    # Calculate Mean Squared Error (MSE)
    train_mse_att = mean_squared_error(y_train_att, y_train_pred_att)
    test_mse_att = mean_squared_error(y_test_att, y_test_pred_att)

    return train_mse_att, test_mse_att

In [268]:
###### IDK IF THIS IS ACTUALLY TELLING ME ANYTHING

# literally all variables, not just the reduced ones

# Set random seed
rand_seed = 34

# define features (X) and target (y)
feats = ['HRk', 'HAvAge', 'HGP', 'HW', 'HL', 'HOL', 'HPTS',
       'HPTS%', 'HGF', 'HGA', 'HSOW', 'HSOL', 'HSRS', 'HSOS', 'HGF/G', 'HGA/G',
       'HPP', 'HPPO', 'HPP%', 'HPPA', 'HPPOA', 'HPK%', 'HSH', 'HSHA', 'HPIM/G',
       'HoPIM/G', 'HS', 'HS%', 'HSA', 'HSV%', 'HSO', 'HCAN', 'VRk', 'VAvAge',
       'VGP', 'VW', 'VL', 'VOL', 'VPTS', 'VPTS%', 'VGF', 'VGA', 'VSOW', 'VSOL',
       'VSRS', 'VSOS', 'VGF/G', 'VGA/G', 'VPP', 'VPPO', 'VPP%', 'VPPA',
       'VPPOA', 'VPK%', 'VSH', 'VSHA', 'VPIM/G', 'VoPIM/G', 'VS', 'VS%', 'VSA',
       'VSV%', 'VSO', 'VCAN', 'TDAY', 'WDAY', 'ThDAY', 'FDAY', 'SDAY', 'SuDAY',
       'DIV', 'CAP', 'LCAP']

dependent_vars = ['A', 'LA', 'PA', 'LPA', 'LA/LC']

cap_index = feats.index('CAP')  # Find the index of 'CAP' in the original features

for dep in dependent_vars:
    X = df_att[feats].values
    y = df_att[dep].values
    
    # initialize model
    model = LinearRegression()
    
    # use k-fold cross-validation (5 folds)
    K = 5
    cv = KFold(n_splits = K, shuffle = True, random_state = rand_seed)
    
    # mse function
    def mse(y, y_pred):
        return -np.mean((y - y_pred)**2)
    
    # scorer object
    mse_scorer = make_scorer(mse, greater_is_better = False)
    
    # compute cross-validation scores (negative MSE for regression)
    cv_scores = cross_val_score(model, X, y, cv = cv, scoring = mse_scorer)
    
    # print average error
    print(dep)
    print('-' * 35)
    print(f'Mean CV Error (MSE): {np.mean(cv_scores)}')
    print(f'Standard Deviation of CV Errors: {np.std(cv_scores)}\n')

    # K-fold cross-validation for train/test splits
    train_mse_att_list = []
    test_mse_att_list = []

    for train_index, test_index in cv.split(X):
        # Split the data into training and testing sets
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        cap_train = X_train[:, cap_index].mean()  # Access 'CAP' column by index in scaled data
        cap_test = X_test[:, cap_index].mean()
        
        # Fit the model to the training data
        model.fit(X_train, y_train)
        
        # Get predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        # Calculate normalized attendance MSE
        train_mse_att, test_mse_att = calc_mse_att(dep, cap_train, cap_test, y_train, y_test, y_train_pred, y_test_pred)
        
        # Append to lists for averaging later
        train_mse_att_list.append(train_mse_att)
        test_mse_att_list.append(test_mse_att)

    # Print average Train and Test MSE for attendance
    print(f"Average Train MSE Attendance: {np.mean(train_mse_att_list)}")
    print(f"Average Test MSE Attendance: {np.mean(test_mse_att_list)}\n")

A
-----------------------------------
Mean CV Error (MSE): 239975.77988280938
Standard Deviation of CV Errors: 13521.365885494984

Average Train MSE Attendance: 230565.25309361727
Average Test MSE Attendance: 239975.77988280938

LA
-----------------------------------
Mean CV Error (MSE): 0.0006262607289974213
Standard Deviation of CV Errors: 3.251195280893399e-05

Average Train MSE Attendance: 227524.02901483784
Average Test MSE Attendance: 236713.23936188064

PA
-----------------------------------
Mean CV Error (MSE): 0.000656464510538592
Standard Deviation of CV Errors: 3.57847803243236e-05

Average Train MSE Attendance: 224726.6424104757
Average Test MSE Attendance: 233845.02443691826

LPA
-----------------------------------
Mean CV Error (MSE): 0.0006262607289973501
Standard Deviation of CV Errors: 3.2511952808893966e-05

Average Train MSE Attendance: 223924.15882216007
Average Test MSE Attendance: 232920.91630082237

LA/LC
-----------------------------------
Mean CV Error (MSE): 6

In [257]:
###### IDK IF THIS IS ACTUALLY TELLING ME ANYTHING

# Set random seed
rand_seed = 34

# define features (X) and target (y)
feats = ['HW', 'HL', 'HSOW', 'HSOL', 'HSOS', 'HPP%', 'HPK%',
       'HSH', 'HSHA', 'HPIM/G', 'HoPIM/G', 'HS%', 'HSV%', 'HSO', 'HCAN', 'VW',
       'VL', 'VSOW', 'VSOL', 'VSOS', 'VPP%', 'VPK%', 'VSH', 'VSHA', 'VPIM/G',
       'VoPIM/G', 'VS%', 'VSV%', 'VSO', 'VCAN', 'TDAY', 'WDAY', 'ThDAY',
       'FDAY', 'SDAY', 'SuDAY', 'DIV', 'CAP', 'LCAP']

dependent_vars = ['A', 'LA', 'PA', 'LPA', 'LA/LC']

cap_index = feats.index('CAP')  # Find the index of 'CAP' in the original features

for dep in dependent_vars:
    X = df_att_feat[feats].values
    y = df_att_feat[dep].values
    
    # initialize model
    model = LinearRegression()
    
    # use k-fold cross-validation (5 folds)
    K = 5
    cv = KFold(n_splits = K, shuffle = True, random_state = rand_seed)
    
    # mse function
    def mse(y, y_pred):
        return -np.mean((y - y_pred)**2)
    
    # scorer object
    mse_scorer = make_scorer(mse, greater_is_better = False)
    
    # compute cross-validation scores (negative MSE for regression)
    cv_scores = cross_val_score(model, X, y, cv = cv, scoring = mse_scorer)
    
    # print average error
    print(dep)
    print('-' * 35)
    print(f'Mean CV Error (MSE): {np.mean(cv_scores)}')
    print(f'Standard Deviation of CV Errors: {np.std(cv_scores)}\n')

    # K-fold cross-validation for train/test splits
    train_mse_att_list = []
    test_mse_att_list = []

    for train_index, test_index in cv.split(X):
        # Split the data into training and testing sets
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        cap_train = X_train[:, cap_index].mean()  # Access 'CAP' column by index in scaled data
        cap_test = X_test[:, cap_index].mean()
        
        # Fit the model to the training data
        model.fit(X_train, y_train)
        
        # Get predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        # Calculate normalized attendance MSE
        train_mse_att, test_mse_att = calc_mse_att(dep, cap_train, cap_test, y_train, y_test, y_train_pred, y_test_pred)
        
        # Append to lists for averaging later
        train_mse_att_list.append(train_mse_att)
        test_mse_att_list.append(test_mse_att)

    # Print average Train and Test MSE for attendance
    print(f"Average Train MSE Attendance: {np.mean(train_mse_att_list)}")
    print(f"Average Test MSE Attendance: {np.mean(test_mse_att_list)}\n")

A
-----------------------------------
Mean CV Error (MSE): 274782.62138197233
Standard Deviation of CV Errors: 18314.080170750836

Average Train MSE Attendance: 268249.589024362
Average Test MSE Attendance: 274782.62138197233

LA
-----------------------------------
Mean CV Error (MSE): 0.000711329815725568
Standard Deviation of CV Errors: 4.3309343438818144e-05

Average Train MSE Attendance: 267377.9158548856
Average Test MSE Attendance: 273759.8592570344

PA
-----------------------------------
Mean CV Error (MSE): 0.0007483224999349977
Standard Deviation of CV Errors: 4.8205648435193876e-05

Average Train MSE Attendance: 260314.88289971286
Average Test MSE Attendance: 266567.6319219131

LPA
-----------------------------------
Mean CV Error (MSE): 0.000711329815725582
Standard Deviation of CV Errors: 4.330934343888156e-05

Average Train MSE Attendance: 260065.7435309876
Average Test MSE Attendance: 266216.23524314247

LA/LC
-----------------------------------
Mean CV Error (MSE): 7.318

In [253]:
# run a linear regression with literally just all the variables lol
# define features (X) and target (y) with all columns except 'A' (attendance)

for dep in dependent_vars:
    X = df_att_feat.drop(columns = ['A', 'LA', 'PA', 'LPA', 'LA/LC', 'H', 'V'])
    y = df_att_feat[dep]
    
    # fit the linear regression model using OLS (Ordinary Least Squares)
    model = sm.OLS(y, X).fit()

    # split the data into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = rand_seed)

    # fit the OLS model on the training data
    model = sm.OLS(y_train, X_train).fit()

    # print model summary
    print(model.summary())
    
    # variables with p < 0.05
    significant_vars = model.pvalues[model.pvalues < 0.05].index.tolist()
    print(f"Variables with p-value < 0.05:\n{significant_vars}\n")

                                 OLS Regression Results                                
Dep. Variable:                      A   R-squared (uncentered):                   0.999
Model:                            OLS   Adj. R-squared (uncentered):              0.999
Method:                 Least Squares   F-statistic:                          1.506e+05
Date:                Mon, 10 Mar 2025   Prob (F-statistic):                        0.00
Time:                        14:50:33   Log-Likelihood:                         -34397.
No. Observations:                4479   AIC:                                  6.887e+04
Df Residuals:                    4440   BIC:                                  6.912e+04
Df Model:                          39                                                  
Covariance Type:            nonrobust                                                  
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------

In [221]:
# # calculate normalized attendance MSE --> so convert everything back to raw attendance numbers
def calc_mse_att(dep, X_train, X_test, y_train, y_test, y_train_pred, y_test_pred):
    if dep == 'A':
        y_train_pred_att = y_train_pred
        y_test_pred_att = y_test_pred

        y_train_att = y_train
        y_test_att = y_test
    
    elif dep == 'LA':
        y_train_pred_att = np.exp(y_train_pred)
        y_test_pred_att = np.exp(y_test_pred)

        y_train_att = np.exp(y_train)
        y_test_att = np.exp(y_test)

    elif dep == 'PA':
        # average capacity
        cap_train = X_train['CAP'].mean()
        cap_test = X_test['CAP'].mean()
        
        y_train_pred_att = y_train_pred * cap_train
        y_test_pred_att = y_test_pred * cap_test

        y_train_att = y_train * cap_train
        y_test_att = y_test * cap_test

    elif dep == 'LPA':
        # average capacity
        cap_train = X_train['CAP'].mean()
        cap_test = X_test['CAP'].mean()
        
        y_train_pred_att = np.exp(y_train_pred) * cap_train
        y_test_pred_att = np.exp(y_test_pred) * cap_test

        y_train_att = np.exp(y_train) * cap_train
        y_test_att = np.exp(y_test) * cap_test

    elif dep == 'LA/LC':
        # average log capacity
        lcap_train = X_train['LCAP'].mean()
        lcap_test = X_test['LCAP'].mean()
    
        y_train_pred_att = np.exp(y_train_pred * lcap_train)
        y_test_pred_att = np.exp(y_test_pred * lcap_test)

        y_train_att = np.exp(y_train * lcap_train)
        y_test_att = np.exp(y_test * lcap_test)
        
    # Calculate Mean Squared Error (MSE)
    train_mse_att = mean_squared_error(y_train_att, y_train_pred_att)
    test_mse_att = mean_squared_error(y_test_att, y_test_pred_att)

    print(f"Train MSE Attendance: {train_mse_att}")
    print(f"Test MSE Attendance: {test_mse_att}\n")

In [227]:
# Initialize the Lasso model with alpha (regularization strength)
lasso = Lasso(alpha = 0.001, max_iter = 100000)

for dep in dependent_vars:
    X = df_att_feat.drop(columns=['A', 'LA', 'PA', 'LPA', 'LA/LC', 'H', 'V'])
    y = df_att_feat[dep]

    # Standardize the data (important for regularization methods like Lasso)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Split into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size = 0.2, random_state = rand_seed)
    X_train_unscaled, X_test_unscaled, _, _ = train_test_split(X, y, test_size = 0.2, random_state = rand_seed)

    # Fit the Lasso model
    lasso.fit(X_train, y_train)

    # Get predictions
    y_train_pred = lasso.predict(X_train)
    y_test_pred = lasso.predict(X_test)

    # Calculate Mean Squared Error (MSE)
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)

    # Get the coefficients and variables with non-zero coefficients
    significant_vars = [feature for feature, coef in zip(X.columns, lasso.coef_) if coef != 0]

    print(f"Lasso regression - Variables with non-zero coefficients for {dep}: {significant_vars}")
    print(f"Train MSE: {train_mse}")
    print(f"Test MSE: {test_mse}\n")

    calc_mse_att(dep, X_train_unscaled, X_test_unscaled, y_train, y_test, y_train_pred, y_test_pred)

Lasso regression - Variables with non-zero coefficients for A: ['HW', 'HL', 'HSOW', 'HSOL', 'HSOS', 'HPP%', 'HPK%', 'HSH', 'HSHA', 'HPIM/G', 'HoPIM/G', 'HS%', 'HSV%', 'HSO', 'HCAN', 'VW', 'VL', 'VSOW', 'VSOL', 'VSOS', 'VPP%', 'VPK%', 'VSH', 'VSHA', 'VPIM/G', 'VoPIM/G', 'VS%', 'VSV%', 'VSO', 'VCAN', 'TDAY', 'WDAY', 'ThDAY', 'FDAY', 'SDAY', 'SuDAY', 'DIV', 'CAP', 'LCAP']
Train MSE: 263287.4613901961
Test MSE: 295491.0185934501

Train MSE Attendance: 263287.4613901961
Test MSE Attendance: 295491.0185934501

Lasso regression - Variables with non-zero coefficients for LA: ['HW', 'HL', 'HSOW', 'HSOL', 'HSOS', 'HPP%', 'HPK%', 'HSH', 'HPIM/G', 'HSV%', 'HCAN', 'VPIM/G', 'VoPIM/G', 'VSV%', 'SDAY', 'SuDAY', 'DIV', 'CAP']
Train MSE: 0.0007460751433821711
Test MSE: 0.0008182474042925108

Train MSE Attendance: 286385.6526594475
Test MSE Attendance: 320758.1716293231

Lasso regression - Variables with non-zero coefficients for PA: ['HW', 'HL', 'HSOW', 'HSOL', 'HSOS', 'HPP%', 'HPK%', 'HSH', 'HPIM/G', 

In [229]:
# Initialize the Ridge model with alpha (regularization strength)
ridge = Ridge(alpha = 1.0)

for dep in dependent_vars:
    X = df_att_feat.drop(columns=['A', 'LA', 'PA', 'LPA', 'LA/LC', 'H', 'V'])
    y = df_att_feat[dep]

    # Standardize the data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Split into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state = rand_seed)
    X_train_unscaled, X_test_unscaled, _, _ = train_test_split(X, y, test_size = 0.2, random_state = rand_seed)

    # Fit the Ridge model
    ridge.fit(X_train, y_train)

    # get predictions
    y_train_pred = ridge.predict(X_train)
    y_test_pred = ridge.predict(X_test)

    # Calculate Mean Squared Error (MSE)
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)

    print(f"Train MSE: {train_mse}")
    print(f"Test MSE: {test_mse}\n")

    calc_mse_att(dep, X_train_unscaled, X_test_unscaled, y_train, y_test, y_train_pred, y_test_pred)

Train MSE: 265132.5182905978
Test MSE: 294182.84400564607

Train MSE Attendance: 265132.5182905978
Test MSE Attendance: 294182.84400564607

Train MSE: 0.0006875668137629401
Test MSE: 0.0007552060461493163

Train MSE Attendance: 263125.7732815497
Test MSE Attendance: 295137.0743120571

Train MSE: 0.00072298754918963
Test MSE: 0.0007982627045527602

Train MSE Attendance: 257664.18769436234
Test MSE Attendance: 283787.72516401205

Train MSE: 0.0006885184775009529
Test MSE: 0.0007553874601809858

Train MSE Attendance: 257209.99220429338
Test MSE Attendance: 284624.9993558693

Train MSE: 7.085425873970411e-06
Test MSE: 7.769166488923908e-06

Train MSE Attendance: 256042.7269347531
Test MSE Attendance: 283012.68693044985



In [231]:
# Initialize the ElasticNet model with alpha (regularization strength) and l1_ratio (balance between Lasso and Ridge)
elastic_net = ElasticNet(alpha = 0.001, l1_ratio = 0.8, max_iter = 200000)

for dep in dependent_vars:
    X = df_att_feat.drop(columns=['A', 'LA', 'PA', 'LPA', 'LA/LC', 'H', 'V'])
    y = df_att_feat[dep]

    # Standardize the data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Split into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state = rand_seed)
    X_train_unscaled, X_test_unscaled, _, _ = train_test_split(X, y, test_size = 0.2, random_state = rand_seed)

    # Fit the ElasticNet model
    elastic_net.fit(X_train, y_train)

    # get predictions
    y_train_pred = elastic_net.predict(X_train)
    y_test_pred = elastic_net.predict(X_test)

    # Calculate Mean Squared Error (MSE)
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)

    # Get the coefficients and variables with non-zero coefficients
    significant_vars = [feature for feature, coef in zip(X.columns, elastic_net.coef_) if coef != 0]
    
    print(f"ElasticNet regression - Variables with non-zero coefficients for {dep}: {significant_vars}\n")
    print(f"Train MSE: {train_mse}")
    print(f"Test MSE: {test_mse}\n")

    calc_mse_att(dep, X_train_unscaled, X_test_unscaled, y_train, y_test, y_train_pred, y_test_pred)

ElasticNet regression - Variables with non-zero coefficients for A: ['HW', 'HL', 'HSOW', 'HSOL', 'HSOS', 'HPP%', 'HPK%', 'HSH', 'HSHA', 'HPIM/G', 'HoPIM/G', 'HS%', 'HSV%', 'HSO', 'HCAN', 'VW', 'VL', 'VSOW', 'VSOL', 'VSOS', 'VPP%', 'VPK%', 'VSH', 'VSHA', 'VPIM/G', 'VoPIM/G', 'VS%', 'VSV%', 'VSO', 'VCAN', 'TDAY', 'WDAY', 'ThDAY', 'FDAY', 'SDAY', 'SuDAY', 'DIV', 'CAP', 'LCAP']

Train MSE: 264891.4475312363
Test MSE: 294138.07198435976

Train MSE Attendance: 264891.4475312363
Test MSE Attendance: 294138.07198435976

ElasticNet regression - Variables with non-zero coefficients for LA: ['HW', 'HL', 'HSOW', 'HSOL', 'HSOS', 'HPP%', 'HPK%', 'HSH', 'HPIM/G', 'HS%', 'HSV%', 'HCAN', 'VPIM/G', 'VoPIM/G', 'VSV%', 'SDAY', 'SuDAY', 'DIV', 'CAP']

Train MSE: 0.0007375074640338439
Test MSE: 0.0008071281875347974

Train MSE Attendance: 282721.2230605397
Test MSE Attendance: 315960.0476063226

ElasticNet regression - Variables with non-zero coefficients for PA: ['HW', 'HL', 'HSOW', 'HSOL', 'HSOS', 'HPP%',

In [234]:
# Loop over dependent variables
for dep in dependent_vars:
    X = df_att_feat.drop(columns=['A', 'LA', 'PA', 'LPA', 'LA/LC', 'H', 'V'])
    y = df_att_feat[dep]

    # Add constant (intercept) to the model
    X = sm.add_constant(X)

    # Standardize the data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Split into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=rand_seed)
    X_train_unscaled, X_test_unscaled, _, _ = train_test_split(X, y, test_size=0.2, random_state=rand_seed)

    # Apply PCA to reduce dimensionality
    pca = PCA()
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)

    # Fit the linear regression model on the PCA components
    pcr_model = LinearRegression()
    pcr_model.fit(X_train_pca, y_train)

    # Get predictions
    y_train_pred = pcr_model.predict(X_train_pca)
    y_test_pred = pcr_model.predict(X_test_pca)

    # Calculate Mean Squared Error (MSE)
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)

    # Print results
    print(f"PCA Regression - MSE for {dep}:")
    print(f"Train MSE: {train_mse}")
    print(f"Test MSE: {test_mse}")

    # Get the number of components used
    explained_variance = pca.explained_variance_ratio_
    num_components = len([var for var in explained_variance if var > 0])  # Number of components explaining variance
    print(f"Number of components used in PCA: {num_components}\n")

    # You can calculate MSE for unscaled data (optional)
    calc_mse_att(dep, X_train_unscaled, X_test_unscaled, y_train, y_test, y_train_pred, y_test_pred)

PCA Regression - MSE for A:
Train MSE: 263287.455883012
Test MSE: 295496.85357404174
Number of components used in PCA: 40

Train MSE Attendance: 263287.455883012
Test MSE Attendance: 295496.85357404174

PCA Regression - MSE for LA:
Train MSE: 0.0006834428998812467
Test MSE: 0.0007584776034747849
Number of components used in PCA: 40

Train MSE Attendance: 262717.5835029981
Test MSE Attendance: 297746.26279413357

PCA Regression - MSE for PA:
Train MSE: 0.0007180441566785593
Test MSE: 0.0008015286661930869
Number of components used in PCA: 40

Train MSE Attendance: 255902.42123345015
Test MSE Attendance: 284948.7963490934

PCA Regression - MSE for LPA:
Train MSE: 0.000683442899881258
Test MSE: 0.0007584776034747961
Number of components used in PCA: 40

Train MSE Attendance: 255718.417476596
Test MSE Attendance: 286103.37403197616

PCA Regression - MSE for LA/LC:
Train MSE: 7.032246536132604e-06
Test MSE: 7.800299462147122e-06
Number of components used in PCA: 40

Train MSE Attendance: 25

In [233]:
# Loop over dependent variables
for dep in dependent_vars:
    X = df_att_feat.drop(columns=['A', 'LA', 'PA', 'LPA', 'LA/LC', 'H', 'V'])
    y = df_att_feat[dep]

    # Add constant (intercept) to the model
    X = sm.add_constant(X)

    # Standardize the data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Split into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=rand_seed)
    X_train_unscaled, X_test_unscaled, _, _ = train_test_split(X, y, test_size=0.2, random_state=rand_seed)

    # Initialize PLS model, choose n_components based on your dataset
    pls_model = PLSRegression(n_components=30)  # You can try different values for n_components
    pls_model.fit(X_train, y_train)

    # Get predictions
    y_train_pred = pls_model.predict(X_train)
    y_test_pred = pls_model.predict(X_test)

    # Calculate Mean Squared Error (MSE)
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)

    # Print results
    print(f"PLS Regression - MSE for {dep}:")
    print(f"Train MSE: {train_mse}")
    print(f"Test MSE: {test_mse}")

    # You can calculate MSE for unscaled data (optional)
    calc_mse_att(dep, X_train_unscaled, X_test_unscaled, y_train, y_test, y_train_pred, y_test_pred)


PLS Regression - MSE for A:
Train MSE: 263287.6029878128
Test MSE: 295531.04619460134
Train MSE Attendance: 263287.6029878128
Test MSE Attendance: 295531.04619460134

PLS Regression - MSE for LA:
Train MSE: 0.0006834432026819826
Test MSE: 0.0007585476577686004
Train MSE Attendance: 262719.14651384554
Test MSE Attendance: 297773.34116800013

PLS Regression - MSE for PA:
Train MSE: 0.000718044376145
Test MSE: 0.0008015239250643075
Train MSE Attendance: 255902.49944868666
Test MSE Attendance: 284947.1108461338

PLS Regression - MSE for LPA:
Train MSE: 0.0006834430933662515
Test MSE: 0.0007584674572735643
Train MSE Attendance: 255718.2409566979
Test MSE Attendance: 286100.07797633996

PLS Regression - MSE for LA/LC:
Train MSE: 7.032248474305757e-06
Test MSE: 7.800197837697694e-06
Train MSE Attendance: 254520.88373394165
Test MSE Attendance: 284452.4710446033

